In [1]:
from pathlib import Path

from tracers.project import setup_project
from tracers.datasets.factory import build_datasets, build_loader_for_model
from tracers.pipelines.trainer_factory import build_trainer

PROJECT_ROOT = Path.cwd().parent
setup = setup_project(PROJECT_ROOT)

MODEL_KEY = "tracer_np"

experiment_config = setup["experiment_config"]
hyperparams = setup["hyperparams"]
segment_frames = setup["segment_frames"]
batch_size = setup["batch_size"]
device = setup["device"]

In [2]:
audio_dir, beatmap_dir, train_dataset, val_dataset, test_dataset, train_ids, val_ids, test_ids = build_datasets(
    PROJECT_ROOT,
    dataset_folder_name="dataset-augment",
    segment_frames=segment_frames,
    seed=42,
)

train_loader = build_loader_for_model(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=0,
    model_key=MODEL_KEY,
)
val_loader = build_loader_for_model(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0,
    model_key=MODEL_KEY,
)
test_loader = build_loader_for_model(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0,
    model_key=MODEL_KEY,
)

trainer, model = build_trainer(
    model_key=MODEL_KEY,
    hyperparams=hyperparams,
    experiment_config=experiment_config,
    device=device,
    checkpoint_path=(
        str(PROJECT_ROOT / experiment_config.resume_from)
        if experiment_config.resume_from is not None
        else None
    ),
    train_loader=train_loader,
    valid_loader=val_loader,
)

if experiment_config.resume_from is not None:
    resume_from = PROJECT_ROOT / experiment_config.resume_from

    # Load pretrained weights, but NOT the old Adam optimizer state.
    trainer.load_checkpoint(
        resume_from,
        load_optimizer=False,
    )

    # Set the fine-tuning learning rate.
    for g in trainer.optimizer.param_groups:
        g["lr"] = hyperparams["learning_rate"]

    # Start a new experiment.
    trainer.start_epoch = 0
    trainer.best_val_loss = float("inf")

    print("Fine-tuning from:", resume_from)
    print("Learning rate:", hyperparams["learning_rate"])
    print("Optimizer state: reset")

Loaded (epoch 5, best val loss 0.1488)
Loaded (epoch 5, best val loss 0.1488)
Fine-tuning from: c:\Users\joshua\vsjycode\csc173\transformers\TRACERS\notebooks\checkpoints\01488\best.pt
Learning rate: 5e-05
Optimizer state: reset


In [3]:
trainer.fit()

wandb: [wandb.login()] Loaded credentials for https://api.forge.coreweave.com from C:\Users\joshua\_netrc.
wandb: Currently logged in as: joshuardh (joshuardh-mindanao-state-university) to https://api.forge.coreweave.com. Use `wandb login --relogin` to force relogin


Validation: 100%|██████████| 151/151 [00:27<00:00,  5.54it/s]



Epoch 1/20

TRAIN
Loss   | Total: 0.0801 | NP: 0.0086 | NS: 0.0544
Onset  | P: 0.8441 | R: 0.8504 | F1: 0.8472
Action | P: 0.9170 | R: 0.8936 | F1: 0.9052
Active Exact Accuracy: 0.4165
GT Active Rate: 4.99% | Predicted Active Rate: 4.87%

VAL
Loss   | Total: 0.0981 | NP: 0.0134 | NS: 0.0580
Onset  | P: 0.7600 | R: 0.7313 | F1: 0.7454
Action | P: 0.8541 | R: 0.8595 | F1: 0.8568
Active Exact Accuracy: 0.3890
GT Active Rate: 4.37% | Predicted Active Rate: 4.40%


Epochs:   5%|▌         | 1/20 [09:37<3:02:44, 577.08s/it]

Saved checkpoint: checkpoints\best.pt
Saved checkpoint: checkpoints\latest.pt


Epochs:  10%|█         | 2/20 [18:24<2:44:21, 547.89s/it]

... (34 outputs left out)


Epoch 19/20

TRAIN
Loss   | Total: 0.0660 | NP: 0.0070 | NS: 0.0449
Onset  | P: 0.8698 | R: 0.8798 | F1: 0.8748
Action | P: 0.9321 | R: 0.9195 | F1: 0.9258
Active Exact Accuracy: 0.4599
GT Active Rate: 4.99% | Predicted Active Rate: 4.92%

VAL
Loss   | Total: 0.1068 | NP: 0.0155 | NS: 0.0602
Onset  | P: 0.7557 | R: 0.7323 | F1: 0.7438
Action | P: 0.8577 | R: 0.8532 | F1: 0.8555
Active Exact Accuracy: 0.3894
GT Active Rate: 4.37% | Predicted Active Rate: 4.35%
Saved checkpoint: checkpoints\latest.pt


Epochs: 100%|██████████| 20/20 [2:56:50<00:00, 530.51s/it]



Epoch 20/20

TRAIN
Loss   | Total: 0.0655 | NP: 0.0069 | NS: 0.0447
Onset  | P: 0.8726 | R: 0.8817 | F1: 0.8772
Action | P: 0.9333 | R: 0.9204 | F1: 0.9268
Active Exact Accuracy: 0.4620
GT Active Rate: 5.00% | Predicted Active Rate: 4.93%

VAL
Loss   | Total: 0.1082 | NP: 0.0160 | NS: 0.0603
Onset  | P: 0.7567 | R: 0.7399 | F1: 0.7482
Action | P: 0.8614 | R: 0.8511 | F1: 0.8562
Active Exact Accuracy: 0.3877
GT Active Rate: 4.37% | Predicted Active Rate: 4.32%
Saved checkpoint: checkpoints\latest.pt


epoch,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
time/epoch_seconds,▃▂▁▂▁▁▁█▁▁▁▁▁▁▁▁▁▁▁▁
time/train_seconds,▃▂▁▂▁▁▁█▁▁▁▁▁▁▁▁▁▁▁▁
time/val_seconds,█▃▃▃▃▃▂▁▁▂▁▁▁▁▁▁▁▄▄▃
train/action_f1,▁▂▂▂▃▃▃▄▄▅▅▅▆▆▆▇▇▇██
train/action_precision,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇▇█
train/action_recall,▁▂▂▂▃▃▃▄▄▅▅▅▆▆▇▇▇▇██
train/active_exact_accuracy,▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇▇██
train/gt_active_rate,▆▇▁█▆▃▅▆▃▄▅▅▃▇▆▇▇▄▄▇
train/np_loss,██▇▇▇▆▆▅▅▅▄▄▃▃▃▂▂▂▁▁
+18,...
